In [ ]:
!pip install -q uv
!uv pip install -U transformers
!uv pip install -q pypdfium2 pillow==10.3.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 44.5 MB/s eta 0:00:00
Using Python 3.12.12 environment at: /usr
Resolved 28 packages in 665ms
Prepared 9 packages in 1.62s
Uninstalled 9 packages in 691ms
Installed 9 packages in 130ms
 - filelock==3.20.3
 + filelock==3.24.0
 - fsspec==2025.3.0
 + fsspec==2026.2.0
 - huggingface-hub==1.4.0
 + huggingface-hub==1.4.1
 - numpy==2.0.2
 + numpy==2.4.2
 - regex==2025.11.3
 + regex==2026.1.15
 - rich==13.9.4
 + rich==14.3.2
 - transformers==5.0.0
 + transformers==5.1.0
 - typer==0.21.1
 + typer==0.23.1
 - typer-slim==0.21.1
 + typer-slim==0.23.1


# Single Page (Image)

In [ ]:
import torch
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32 if device == "mps" else torch.bfloat16

model = LightOnOcrForConditionalGeneration.from_pretrained("lightonai/LightOnOCR-2-1B", torch_dtype=dtype).to(device)
processor = LightOnOcrProcessor.from_pretrained("lightonai/LightOnOCR-2-1B")

# url = "https://huggingface.co/datasets/hf-internal-testing/fixtures_ocr/resolve/main/SROIE-receipt.jpeg"
# url = "/content/Screenshot from 2026-02-12 00-55-07.png"
# url = "/content/Corporate Governance Report for the year 2025 Arabic/auto/images/622fd8b969082ecf84e852806e96a29a4f2fb626e2f370d35f4f3126b03aca48.jpg"
# url = "/content/Corporate Governance Report for the year 2025 Arabic/auto/images/c59d95a7d89ba759c81150322a6582b06a70262e5b738d22f19a1b42bb6b84b1.jpg"
url = "/content/Corporate Governance Report for the year 2025 Arabic/auto/images/4db6dc33adbc38eecb3a2f456d3d8d95222c2f9e21b89a8f026d77942b3ca544.jpg"

conversation = [{"role": "user", "content": [{"type": "image", "url": url}]}]

inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)
inputs = {k: v.to(device=device, dtype=dtype) if v.is_floating_point() else v.to(device) for k, v in inputs.items()}

output_ids = model.generate(**inputs, max_new_tokens=1024)
generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
# output_text = processor.decode(generated_ids, skip_special_tokens=True)
output_text = processor.decode(generated_ids, skip_special_tokens=False)

print(output_text)


## Full file (PDF)

In [ ]:
import torch
import pypdfium2 as pdfium
from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32 if device == "mps" else torch.bfloat16

model = LightOnOcrForConditionalGeneration.from_pretrained("lightonai/LightOnOCR-2-1B", torch_dtype=dtype).to(device)
processor = LightOnOcrProcessor.from_pretrained("lightonai/LightOnOCR-2-1B")



pdf_path = '/content/drive/MyDrive/Click ITS/ATE Project/Docs/nbe-document-28.9.pdf'

# Convert PDF to images
pdf = pdfium.PdfDocument(pdf_path)

all_text = []

for i in range(len(pdf)):
    page = pdf[i]

    # Render page to PIL image
    bitmap = page.render(scale=2)  # <-- correct call on PAGE, not document
    pil_image = bitmap.to_pil()

    conversation = [{
        "role": "user",
        "content": [{"type": "image", "image": pil_image}]
    }]

    inputs = processor.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    inputs = {
        k: v.to(device=device, dtype=dtype) if v.is_floating_point()
        else v.to(device)
        for k, v in inputs.items()
    }

    output_ids = model.generate(**inputs, max_new_tokens=2048)
    generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]

    text = processor.decode(generated_ids, skip_special_tokens=True)

    print(f"\n===== Page {i+1} =====\n")
    print(text)

    all_text.append(text)

full_text = "\n\n".join(all_text)
